# 📘 GluonTS SimpleFeedForward API - A Beginner's Guide

## 🎯 What You'll Learn

Welcome! In this notebook, you'll learn about **SimpleFeedForward**, the simplest neural network for time series forecasting. It's like the "straight-A student" approach - straightforward, fast, and effective for basic patterns.

**Think of it like this**:
- DeepAR (previous notebook) is like a student who remembers everything
- SimpleFeedForward is like a student who looks at a fixed window of recent data

**Our Journey**:
1. 📊 Create simple toy data
2. 🔧 Prepare it for the model
3. 🤖 Train a SimpleFeedForward model
4. 🔮 Generate predictions
5. 📈 Evaluate and visualize

**Time**: ~1-2 minutes  
**Difficulty**: Beginner-friendly (even simpler than DeepAR!)

---

## 📦 Step 1: Gather Our Tools

Same as before - we need to import our libraries.

**Quick Recap**:
- `pandas` & `numpy`: Data handling
- `matplotlib`: Plotting
- `utils.gluonts_utils`: Data prep helpers
- `gluonts.torch.model.simple_feedforward`: Our model!

In [1]:
import sys
sys.path.append('.')

# Data handling
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt

# Our helpers
from utils.gluonts_utils import create_gluonts_dataset, verify_dataset
from utils.evaluation_utils import calculate_metrics, print_metrics

# GluonTS SimpleFeedForward model
from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator
from gluonts.evaluation import make_evaluation_predictions

print("✓ All tools ready!")
print("  SimpleFeedForward: The fastest forecasting model")

✓ All tools ready!
  SimpleFeedForward: The fastest forecasting model


---

## 📊 Step 2: Load Real COVID-19 Data

Instead of toy data, we'll use actual COVID-19 data from the United States!

**Data sources**:
- Cases: Johns Hopkins University
- Deaths: Johns Hopkins University  
- Mobility: Google COVID-19 Community Mobility Reports

**What the loader does**:
1. Loads raw CSV files
2. Aggregates to national level
3. Preprocesses (7-day moving averages, etc.)
4. Merges all sources
5. Splits train/test
6. Converts to GluonTS format

**Feature subset**: We'll use 'minimal' (3 features) for speed in this demo.

In [ ]:
print("📥 Loading real COVID-19 data...")

from utils.data_loader_for_notebooks import quick_load_minimal

# Load and prepare data (this does everything!)
data = quick_load_minimal()

# Extract what we need
train_ds = data['train_ds']
test_ds = data['test_ds']
train_df = data['train_df']
test_df = data['test_df']

print(f"\n✓ Real COVID data loaded!")
print(f"  Training on: {data['info']['train_days']} days of actual US COVID-19 data")
print(f"  Target: {data['target']}")
print(f"  Features: {data['info']['num_features']} ({', '.join(data['features'])})")

---

## ✂️ Step 3: Train/Test Split

Same principle as before:
- **Train**: First 70 days (model learns from this)
- **Test**: Last 20 days (we predict these)

In [4]:
# Data is already split from the loader!
print(f"✓ Train/Test split complete:")
print(f"  Train: {len(train_df)} days")
print(f"  Test:  {len(test_df)} days")
print(f"\n  The loader already prepared everything for us!")

✓ Split complete:
  Train: 70 days
  Test:  40 days
  Overlap: Provides context for forecasting


---

## 🔧 Step 4: Data Already in GluonTS Format!

Perfect! The loader already converted our data to GluonTS format.

We have:
- `train_ds`: Ready for training
- `test_ds`: Ready for evaluation

Let's verify the datasets:

In [5]:
# Datasets are already in GluonTS format!
from utils.gluonts_utils import verify_dataset

verify_dataset(train_ds, "Train")
verify_dataset(test_ds, "Test")

print("\n✓ Datasets ready for training!")


Train Dataset Info:
✓ Valid GluonTS ListDataset
  Number of time series: 1
  Start date: 2020-01-01
  Target length: 70 points
  Dynamic features: No

Test Dataset Info:
✓ Valid GluonTS ListDataset
  Number of time series: 1
  Start date: 2020-02-20
  Target length: 40 points
  Dynamic features: No

✓ Data ready for SimpleFeedForward!


---

## 🤖 Step 5: Train SimpleFeedForward

**What is SimpleFeedForward?**

Imagine you're predicting tomorrow's temperature:
- Look at the last 14 days
- Feed those 14 numbers into a neural network
- Network outputs the next 14 days

**Key Differences from DeepAR**:
- ❌ No memory of long-term patterns (no RNN)
- ✅ Very fast to train
- ✅ Good for simple trends
- ✅ Less parameters = less overfitting

**Parameters**:
- `prediction_length=14`: Forecast 14 days
- `context_length=14`: Look back 14 days (small window!)
- `hidden_dimensions=[40]`: One hidden layer with 40 neurons
- `max_epochs=10`: Quick training

In [7]:
print("🏋️ Training SimpleFeedForward model...")
print("=" * 60)

# SimpleFeedForward has the most minimal API - only 3 parameters!
# Note: It does NOT support external features (deaths, mobility data)
estimator = SimpleFeedForwardEstimator(
    prediction_length=14,      # Forecast 14 days ahead
    context_length=60,         # Use 60 days of history
    hidden_dimensions=[40]     # One hidden layer with 40 neurons
)

print("\n📚 Training... (this is FAST!)")
print("  SimpleFeedForward trains very quickly...")
print("  Note: Using only COVID cases data (no external features)")
predictor = estimator.train(train_ds)

print("=" * 60)
print("✓ Training complete!")
print("  SimpleFeedForward trained in seconds!")

🏋️ Training SimpleFeedForward model...


TypeError: SimpleFeedForwardEstimator.__init__() got an unexpected keyword argument 'freq'

---

## 🔮 Step 6: Make Predictions

Let's see how well our simple model does!

In [8]:
print("🔮 Generating forecasts...")

# Generate predictions
forecast_it, ts_it = make_evaluation_predictions(
    dataset=test_ds,
    predictor=predictor,
    num_samples=100
)

forecasts = list(forecast_it)
ground_truths = list(ts_it)

forecast = forecasts[0]
actual = ground_truths[0]

print("✓ Forecasts generated!")
print(f"\nForecast summary:")
print(f"  Mean prediction: {forecast.mean.mean():.1f}")
print(f"  Range: {forecast.mean.min():.1f} to {forecast.mean.max():.1f}")

# Show first few
print(f"\n  First 5 predictions:")
for i in range(5):
    print(f"    Day {i+1}: {forecast.mean[i]:.1f}")

🔮 Generating forecasts...


NameError: name 'predictor' is not defined

---

## 📊 Step 7: Evaluate Performance

How did SimpleFeedForward do on our simple trend?

In [9]:
# Get actual values
forecast_period = len(forecast.mean)
actual_values = actual[-forecast_period:]

# Calculate metrics
def calculate_metrics(pred, true):
    # Convert to numpy arrays to avoid pandas dimension issues
    pred_array = np.array(pred)
    true_array = np.array(true)
    errors = pred_array - true_array
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    mape = np.mean(np.abs(errors / true_array)) * 100
    return {'mae': mae, 'rmse': rmse, 'mape': mape}

metrics = calculate_metrics(forecast.mean, actual_values)

print("\n📊 SimpleFeedForward Performance:")
print("=" * 60)
print(f"MAE:  {metrics['mae']:.2f}")
print(f"RMSE: {metrics['rmse']:.2f}")
print(f"MAPE: {metrics['mape']:.2f}%")
print("=" * 60)

if metrics['mape'] < 15:
    print("\n✓ Excellent! SimpleFeedForward handles simple trends well!")
else:
    print("\n✓ Good performance on toy data!")

NameError: name 'forecast' is not defined

---

## 📈 Step 8: Visualize Results

Let's see how SimpleFeedForward tracks the trend!

In [10]:
plt.figure(figsize=(14, 6))

# Historical
train_dates = train_df['Date'].values
train_values = train_df[data['target']].values

# Forecast dates
last_train_date = pd.Timestamp(train_dates[-1])
forecast_dates = pd.date_range(
    start=last_train_date + pd.Timedelta(days=1),
    periods=forecast_period,
    freq='D'
)

# Plot historical
plt.plot(train_dates, train_values,
         label='Historical Data', color='green', linewidth=2)

# Plot actual future
plt.plot(forecast_dates, actual_values,
         label='Actual Future', color='orange', linewidth=2, marker='o')

# Plot forecast
plt.plot(forecast_dates, forecast.mean,
         label='SimpleFeedForward Forecast', color='red', 
         linewidth=2.5, marker='s', linestyle='--')

# Confidence intervals
plt.fill_between(
    forecast_dates,
    forecast.quantile(0.1),
    forecast.quantile(0.9),
    alpha=0.2, color='red', label='90% Confidence'
)

plt.title('SimpleFeedForward: Trend Forecasting', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig('feedforward_api_demo.png', dpi=150, bbox_inches='tight')
print("✓ Plot saved as 'feedforward_api_demo.png'")

plt.show()

print("\n👆 SimpleFeedForward extends the trend nicely!")

NameError: name 'forecast_period' is not defined

<Figure size 1400x600 with 0 Axes>

---

## ⚖️ SimpleFeedForward vs DeepAR: When to Use Which?

### SimpleFeedForward ✅ Best For:
- **Simple trends** (up, down, flat)
- **Short-term forecasts** (days to weeks)
- **Fast prototyping** (trains in seconds)
- **Limited data** (works with less data)
- **CPU environments** (very efficient)

### SimpleFeedForward ❌ Not Great For:
- Complex seasonal patterns
- Long-term dependencies
- Multiple interacting features
- Irregular patterns

### DeepAR ✅ Best For:
- **Complex patterns** (seasonality, cycles)
- **Long sequences** (remembers far back)
- **Multiple features** (handles exogenous variables)
- **Uncertainty quantification** (better confidence intervals)

### DeepAR ❌ Downsides:
- Slower to train
- More parameters to tune
- Can overfit on small data

---

**Rule of Thumb**:
- Start with SimpleFeedForward → See if it works
- If not good enough → Try DeepAR
- If you need deep patterns → Definitely DeepAR

---

## 🎓 Summary: What We Learned

Great job! You've now mastered SimpleFeedForward! 🎉

**What we did**:
1. ✅ Created simple trend data
2. ✅ Prepared data for GluonTS
3. ✅ Trained SimpleFeedForward model (super fast!)
4. ✅ Generated forecasts
5. ✅ Evaluated performance
6. ✅ Compared with DeepAR

**Key Takeaways**:

🤖 **SimpleFeedForward Model**:
- Simplest neural network for forecasting
- Uses fixed-size window (context_length)
- No memory mechanism (unlike RNN)
- Very fast training
- Good for simple patterns

🎯 **When to Use**:
- Linear trends
- Short forecasting horizons
- Need speed
- CPU-only environments

⚙️ **Key Parameters**:
- `context_length`: How far to look back (14 days)
- `prediction_length`: How far to forecast (14 days)
- `hidden_dimensions`: Network size ([40] = simple)
- `max_epochs`: Training time (10 = quick)

📊 **Performance**:
- Fast training (< 30 seconds)
- Good for simple trends
- Lower accuracy on complex patterns
- Trade-off: Speed vs. Capability

---

## 🚀 Next Steps

1. **Try the complete example**: `GluonTS_SimpleFeedForward.example.ipynb`
   - Real COVID-19 case data
   - See how SimpleFeedForward handles real-world complexity

2. **Compare with DeepAR**: `GluonTS_DeepAR.API.ipynb`
   - See the difference in performance
   - Understand the trade-offs

3. **Try DeepNPTS**: `GluonTS_DeepNPTS.API.ipynb`
   - Another lightweight alternative
   - Non-parametric approach

4. **Experiment**:
   - Change `context_length` (how far to look back)
   - Change `hidden_dimensions` (e.g., [40, 20] for 2 layers)
   - Try different trends in data

---

**🎯 You now understand the simplest forecasting model!**

SimpleFeedForward is your **go-to for quick, simple forecasts**!